## Session 2: Specialist vs Generalist Agents

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

model = ChatOpenAI(model = "gpt-5.4-mini-2026-03-17" , temperature = 0)

## Part 1: Generalist Agent

One node. One prompt. It tries to answer every kind of query by itself.

In [ ]:
class GeneralistState(TypedDict):

    query: str
    answer: str

In [ ]:
def generalist(state: GeneralistState):

    query = state['query']

    prompt = f'You are a helpful support assistant. Answer this user query:\n{query}'
    answer = model.invoke(prompt).content

    return {'answer': answer}

In [ ]:
graph = StateGraph(GeneralistState)

# nodes
graph.add_node('generalist', generalist)

# edges
graph.add_edge(START, 'generalist')
graph.add_edge('generalist', END)

generalist_workflow = graph.compile()

In [ ]:
generalist_workflow

In [ ]:
queries = [
    'Why was I charged twice on my last invoice?',
    'The app crashes every time I upload a photo',
    'What are your support hours?'
]

for query in queries:
    result = generalist_workflow.invoke({'query': query})
    print('QUERY:', query)
    print('ANSWER:', result['answer'])
    print('-' * 60)

## Part 2: Specialist Agents

A router first decides the category. Then the query goes to the matching specialist.

In [ ]:
class RouteSchema(BaseModel):

    category: Literal["billing", "technical", "general"] = Field(description='Which specialist should handle this query')

In [ ]:
structured_model = model.with_structured_output(RouteSchema)

In [ ]:
class SpecialistState(TypedDict):

    query: str
    category: Literal["billing", "technical", "general"]
    answer: str

In [ ]:
def classify_query(state: SpecialistState):

    prompt = f'''Classify this support query into one category: billing, technical, or general.

billing = payments, invoices, refunds, charges
technical = bugs, crashes, login, app not working
general = hours, policies, anything else

Query: {state["query"]}'''

    category = structured_model.invoke(prompt).category

    return {'category': category}


def route_query(state: SpecialistState) -> Literal["billing_specialist", "technical_specialist", "general_specialist"]:

    if state['category'] == 'billing':
        return 'billing_specialist'
    elif state['category'] == 'technical':
        return 'technical_specialist'
    else:
        return 'general_specialist'


def billing_specialist(state: SpecialistState):

    prompt = f'''You are a billing specialist.
Help with invoices, refunds, and charges.
Always mention that this reply is from the Billing Team.

Query: {state["query"]}'''

    answer = model.invoke(prompt).content

    return {'answer': answer}


def technical_specialist(state: SpecialistState):

    prompt = f'''You are a technical support specialist.
Help with bugs, crashes, and app issues.
Always mention that this reply is from Tech Support.
Give short practical steps.

Query: {state["query"]}'''

    answer = model.invoke(prompt).content

    return {'answer': answer}


def general_specialist(state: SpecialistState):

    prompt = f'''You are a general helpdesk specialist.
Help with hours, policies, and basic questions.
Always mention that this reply is from the Helpdesk.

Query: {state["query"]}'''

    answer = model.invoke(prompt).content

    return {'answer': answer}

In [ ]:
graph = StateGraph(SpecialistState)

# nodes
graph.add_node('classify_query', classify_query)
graph.add_node('billing_specialist', billing_specialist)
graph.add_node('technical_specialist', technical_specialist)
graph.add_node('general_specialist', general_specialist)

# edges
graph.add_edge(START, 'classify_query')
graph.add_conditional_edges('classify_query', route_query)
graph.add_edge('billing_specialist', END)
graph.add_edge('technical_specialist', END)
graph.add_edge('general_specialist', END)

specialist_workflow = graph.compile()

In [ ]:
specialist_workflow

In [ ]:
for query in queries:
    result = specialist_workflow.invoke({'query': query})
    print('QUERY:', query)
    print('CATEGORY:', result['category'])
    print('ANSWER:', result['answer'])
    print('-' * 60)